In [ ]:
import torch, torchvision
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split




## Load data

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
len(X) # * .12

569

In [70]:
# Split
x_train, x_val, y_train, y_val = train_test_split(X, y, test_size= .12, random_state = 0)

# Scale
scalar = StandardScaler()
x_train = scalar.fit_transform(x_train)
x_val = scalar.transform(x_val)

len(x_train), len(x_val)

(500, 69)

## Prepare data batches

In [71]:
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(2)

x_train_tensor = torch.from_numpy(x_train).float()
x_val_tensor = torch.from_numpy(x_val).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_val_tensor = torch.from_numpy(y_val).float().unsqueeze(1)


training_loader = DataLoader(TensorDataset(x_train_tensor, y_train_tensor),
                              batch_size = 64, shuffle = True)



## Build Neural Network

In [72]:
# Imports
from torch import nn
from torch.nn import functional as F

# Build the class
class NeuralNetwork(nn.Module):

  def __init__(self):
    super().__init__()

    self.fc1 = nn.Linear(30, 32)
    self.fc2 = nn.Linear(32, 8)
    self.fc3 = nn.Linear(8, 1)
    self.dropout = nn.Dropout(.3)

  def forward(self, x):

    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.dropout(x)
    x = F.sigmoid(self.fc3(x))

    return x





## Inspect the model

In [61]:
# # ! pip install torchinfo

# from torchinfo import summary


# model = NeuralNetwork()
# # summary(model, input_size=(32, 30))

## Seting the model & hyper parameters

In [73]:
from torch.optim import Adam
torch.manual_seed(2)

model = NeuralNetwork()
optimizer = Adam(model.parameters(), lr= .001)
criterion = nn.BCELoss()

epochs = 30
patience = 6
p_counter = 0
best_val_acc = 0
model_path = 'model_breast_cancer.pth'

# calculating acc for Binary tasks
def calc_acc(model, x_test, y_test):
  with torch.no_grad():
    model.eval()
    pred = model(x_test)
    acc = ((pred >= .5) == y_test).float().mean().item()
    return acc


## Training loop

In [74]:
torch.manual_seed(2)


for epoch in range(1, epochs + 1):
  total_loss = 0
  model.train()

  for features, targets in training_loader:

    # Calc outputs
    outputs = model(features)

    # Calc loss & update weights
    loss = criterion(outputs, targets)
    total_loss += loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  # Show progress
  Message = f'epoch: {str(epoch).zfill(2)}/{epochs}, loss:{(total_loss / len(training_loader)).item():.4f}'
  print(Message)


  # Checkpoint
  acc = calc_acc(model, x_val_tensor, y_val_tensor)

  if acc > best_val_acc:
    torch.save(model.state_dict(), model_path)

    p_counter = 0
    best_val_acc = acc
    print(f'New model has been saved with acc = {acc:.4f}\n')
  else:
    p_counter +=1


  # Earlystoping

  if patience == p_counter:
    print(f'Earlystoping at epoch {epoch}, with val_acc: {best_val_acc}')
    break









epoch: 01/30, loss:0.7437
New model has been saved with acc = 0.4203

epoch: 02/30, loss:0.7085
epoch: 03/30, loss:0.6797
New model has been saved with acc = 0.4638

epoch: 04/30, loss:0.6428
New model has been saved with acc = 0.7101

epoch: 05/30, loss:0.5917
New model has been saved with acc = 0.8406

epoch: 06/30, loss:0.5384
New model has been saved with acc = 0.9130

epoch: 07/30, loss:0.4818
New model has been saved with acc = 0.9275

epoch: 08/30, loss:0.4267
New model has been saved with acc = 0.9565

epoch: 09/30, loss:0.3562
epoch: 10/30, loss:0.3078
epoch: 11/30, loss:0.2633
epoch: 12/30, loss:0.2331
epoch: 13/30, loss:0.1877
epoch: 14/30, loss:0.1775
New model has been saved with acc = 0.9855

epoch: 15/30, loss:0.1664
epoch: 16/30, loss:0.1452
epoch: 17/30, loss:0.1322
epoch: 18/30, loss:0.1285
epoch: 19/30, loss:0.1151
epoch: 20/30, loss:0.1201
New model has been saved with acc = 1.0000

epoch: 21/30, loss:0.1177
epoch: 22/30, loss:0.0998
epoch: 23/30, loss:0.1012
epoch:

## Evaluating the model

In [75]:
model.load_state_dict(torch.load(model_path))

calc_acc(model, x_val_tensor, y_val_tensor)

1.0

## Saving the model

In [76]:
torch.save(model.state_dict(), 'model_breast_cancer.pth')



In [77]:
model = NeuralNetwork()
print(calc_acc(model, x_val_tensor, y_val_tensor))

model.load_state_dict(torch.load('model_breast_cancer.pth'))
calc_acc(model, x_val_tensor, y_val_tensor)

0.5797101259231567


1.0